# EE / CprE / CybE 3250: Machine Learning for Electrical, Computer, and Cybersecurity Engineering
## Lab-05: Sequence Modeling with Recurrent Neural Networks
> *[Note from your instructor: So far, we have worked with models designed for tabular data and spatial data. However, many real-world problems involve sequential data, where the order of inputs matters. Examples include time series measurements, sensor readings, and network traffic over time. Traditional models struggle in this setting because they do not account for dependencies between observations. In this lab, we introduce Recurrent Neural Networks (RNNs), which are specifically designed to process sequences. We will apply RNNs to sequential data and compare their behavior to the models we have used previously. You will implement and train several RNNs in PyTorch, gaining insight into how temporal dependencies can be modeled, and where these architectures can be successful. As before, collaboration is encouraged, as is the use of generative AI for debugging and coding support. However, all conceptual answers must be your own work. You will be evaluated based on lab preparation (PreLab), responses to programming and conceptual questions (Lab), and demonstration of understanding (PostLab).]*

**Names:** 

**Group Name:**



### 0. Prelab
Please read through this entire document and answer the questions at the end under the “Prelab Questions” section header. These are due and will be checked at the beginning of the lab – please post if you have any questions.


In [ ]:
# Initial setup
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from d2l import torch as d2l

### 1. Engineering Sequential Data
We have made significant progress in working with *fixed* data, where each sample's features could be represented in a single vector, $\mathbf{x} \in \mathbb{R}^d$, to be processed independently. Many other data sources are inherently *sequential* in nature, where instead of working with independent samples, our data is organized into ordered lists of feature vectors $(x_1, ..., x_T)$. The MLPs and CNNs we have explored so far do not explicitly model temporal dependencies, making them less effective for many sequential tasks.

Sequential data gives rise to a variety of interesting machine learning problems. For example, when analyzing sensor measurements that were taken over a period of time, we may wish to classify the current state of a system based on its recent history, or predict future values from past observations. Many other important applications are also sequential in nature, including speech, language, and text processing. In this lab, we will explore models designed specifically to capture these temporal relationships and leverage information from previous observations when making predictions.

We'll start with the [Predictive Maintenance Dataset](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) from the UCI Machine Learning Repository. This dataset is synthetically generated, but is designed to reflect realistic predictive maintenance data from an industrial machine. It contains 10,000 observations collected from a single milling machine over time. Each observation includes measurements of air temperature, process temperature, rotational speed, torque, and tool wear (measured as the cumulative operating time of the tool in minutes). These variables make up the feature set used by the model.

The dataset contains six label columns. The first, `Machine Failure`, indicates whether any failure occurred (`0=` no failure, `1=` failure). The remaining five labels represent specific failure modes:

- `TWF` (Tool Wear Failure): The cutting tool reached the end of its useful life or physically failed.
- `HDF` (Heat Dissipation Failure): The difference between the air and process temperatures fell below a critical threshold, indicating insufficient heat dissipation.
- `PWF` (Power Failure): The power output, determined by the combination of torque and rotational speed, fell outside acceptable operating limits.
- `OSF` (Overstrain Failure): Excessive tool wear combined with high torque produced mechanical strain beyond safe limits.
- `RNF` (Random Failure): A failure caused by background factors not captured by the measured features.

Our objective is to predict whether a machine failure has occurred and, if so, identify the corresponding failure mode. Since each failure type is represented by its own binary label, this problem can be viewed as a collection of binary classification tasks. Before building our models, we will begin by exploring the dataset and its characteristics.

In [ ]:
maintenance_df = pd.read_csv('data/ai4i2020.csv')
maintenance_df.head(30)

Notice that this dataset appears in the familiar row-wise format we have used throughout the course. You might have observed that this is not actually a true sequential dataset. To see why, ask yourself: *what sequence has been provided*? At first glance, it may seem that we have been given a single continuous timeline of measurements from the milling machine. The `Tool wear [min]` column, which generally increases over time, reinforces this intuition.

However, the dataset is fundamentally organized as a collection of independent examples, each with its own features and labels. In its current form, it is no different from the tabular datasets we have previously analyzed, and we could reasonably apply a standard row-based classifier. The challenge is that this representation does not explicitly capture temporal behavior, such as trends, rates of change, or the accumulation of effects over time.
Our goal in this lab is to explore Recurrent Neural Networks (RNNs), which are designed to learn from sequences rather than individual observations. To take advantage of these models, we will need to think carefully about how to transform our data into a form that exposes its temporal structure. Keep this question in mind as we continue exploring the dataset.

To understand machine failures, it is helpful to approach the dataset from an engineering perspective. Consider the failure type `PWF` (Power Failure). Recall that mechanical power is related to torque and rotational speed:

$$ P = \tau \cdot \omega $$

- **Q1.1)\:**  Based on this relationship, what combinations of torque and rotational speed would you expect to be associated with power failures? Would you expect failures to occur when power is too low, too high, or both?

- **A1.1)\:** 

To test your hypothesis, complete the code below to create a scatter plot of Torque versus Rotational Speed. Plot healthy operating points and power-failure points using different colors. Use Rotational speed on the x-axis and Torque on the y-axis.

In [ ]:
# Create the figure matching your dimensions
plt.figure(figsize=(10, 6))

# Separate your healthy points and power failures by selecting on `PWF` column

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

plt.legend(title='Machine Status', frameon=True, facecolor='white', edgecolor='none')

plt.tight_layout()
plt.show()

- **Q1.2:** What patterns do you observe (what are the conditions where `PWF` is likely?) Was your prediction correct?

- **A1.2:** 

Since determining failure modes is a classification problem, we will want to observe the class balances of our labels. Unlike in prior examples, in this dataset, multiple classes can be true at the same time, as more than one failure can occur simultaneously. In the code block below, finish the bar chart for the class distribution by defining and printing out `failure_counts`:

In [ ]:
# Define the 5 specific engineering failure columns
failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']


# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

# Create the bar chart
plt.figure(figsize=(10, 6))
plt.bar(failure_counts.index, failure_counts.values, color=plt.cm.inferno(np.linspace(0.2, 0.8, len(failure_counts.values))), alpha=0.7)

# Add values on top of each bar for explicit clarity
for i, val in enumerate(failure_counts.values):
    plt.text(i, val + 2, str(val), ha='center', va='bottom', fontweight='bold')

plt.title('Distribution of Specific Mechanical Failure Modes (AI4I 2020)', fontsize=14)
plt.xlabel('Failure Mode Type', fontsize=12)
plt.ylabel('Total Number of Failures', fontsize=12)
plt.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

This shows the distribution of failures, given that a failure did indeed occur. However, we also want to observe the distribution of binary cases where a failure did or did not occur at all:

In [ ]:

# Count the occurrences of overall machine failures (0 = Success, 1 = Failure)
binary_counts = maintenance_df['Machine failure'].value_counts()

print("Overall Class Distribution:")
print(binary_counts)

# Calculate percentages to display as additional context
total_samples = len(maintenance_df)
percentages = (binary_counts / total_samples) * 100

plt.figure(figsize=(8, 6))

# Map specific colors for normal vs failure
colors = ['slategrey', 'crimson']

bars = plt.bar(['Normal Operation (0)', 'Machine Failure (1)'], 
    binary_counts.values, color=colors, edgecolor='none', width=0.5)

# Annotate each bar 
for bar, pct in zip(bars, percentages.values):
    yval = bar.get_height()
        
    label_text = f"{int(yval):,} ({pct:.1f}%)"
    
    plt.text(
        bar.get_x() + bar.get_width()/2, yval + 150, label_text, 
        ha='center', va='bottom', fontweight='bold', fontsize=11
    )

plt.title('Normal vs. Failed Machine States', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Total Number of Records', fontsize=12)


plt.ylim(0, max(binary_counts.values) * 1.15)
plt.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

Here we can clearly see the severe class imbalance in the dataset: the machine operates normally the vast majority of the time. As a result, a model could simply predict no failure for every example and still achieve nearly 97% accuracy; we have seen this story before, and know that performance as it pertains to accuracy and loss values alone won't be enough to access the model. Through confusion matrices, we can get a more full picture of our model's success classifying each label. 

We are now ready to construct our data class, which requires us to revisit an important question: how should we represent the data as a sequence? One natural approach is to divide the dataset into shorter, fixed-length sequences. The model can then process each sequence as a unit and make a prediction based on the information contained within that block.

To support this, we will add a `seq_len` parameter to the `__init__` function. This parameter specifies the number of consecutive observations that make up each sequence, allowing us to control how much historical information is provided to the model for each prediction. For each value `X` and `y`, the model should be provided with the current observation along with a fixed number `seq_len` of preceding observations. This is referred to as a *sliding window* representation.

Note that in doing this we are assuming that the rows in our tabular data have some underlying temporal structure. This is somewhat reasonable, as our `Tool wear` increases over time and the temperature values were generated using [Random walks](https://en.wikipedia.org/wiki/Random_walk). Still, this is a choice we are making when modeling this data, vs. a property that is explicitly guaranteed by the dataset. 

- **Q1.3:** Complete the `__init__` function below, and explain your implementation of the sliding window of length `seq_len`. For this, `X_seq` and `y_multi` lists are provided and will be filled in through your `for` loop implementation using the `.append` function. The tricky part is ensuring your `for` loop condition is set up properly; for this, consider how to ensure you loop through all possible blocks of `seq_len` without running out of room. 

- **A1.3:** 

In [ ]:
class MaintenanceSequenceDataset(d2l.DataModule):
    def __init__(self, df, seq_len=10, batch_size=64):
        super().__init__()
        self.save_hyperparameters()

        # Select continuous engineering features
        feature_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

        # Select the 5 multi-label failure targets
        target_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

        # Main tasks to complete for students:
        # 1. Grab the engineering features and failure targets from the df DataFrame
        # 2. Standardize the features for numerical stability
        # 3. Build sliding window history sequences. The sequences should be of length `seq_len` and the target should be the failure state at the final step of the window.
        
        X_seq, y_multi = [], []          

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

        self.num_train = int(len(self.X) * 0.8)
        
    def get_dataloader(self, train):
        i = slice(0, self.num_train) if train else slice(self.num_train, None)
        return self.get_tensorloader((self.X, self.y), train, i)

Test that our dataloaders are working properly by loading a batch of training data:

In [ ]:
MaintenanceData = MaintenanceSequenceDataset(df=maintenance_df, seq_len=10, batch_size=64)
X, y = next(iter(MaintenanceData.train_dataloader()))
print(X.shape, y.shape)

- **Q1.4:** A correct implementation of `MaintenanceSequenceDataset` should generate tensors of size `([batch_size, sequence_length, num_features])` and `([batch_size, num_targets])`. How does this compare to when we previously were exploring tabular data for classification? Briefly explain. 

- **A1.4:** 

### 2. Introduction to Recurrent Neural Networks (RNNs)

Recurrent Neural Networks (RNNs) are designed to process sequential data, handling time-step by time-step, rather than processing an entire sample in a single pass as MLPs and CNNs do. As each element of a sequence is processed, the model updates an internal hidden state, which serves as a form of memory. This hidden state is carried forward through the sequence, allowing the model to learn temporal relationships and trends that span multiple observations.


To understand how recurrent layers work, it is helpful to start with the models we have already studied. Recall that a dense (fully connected) layer computes:


$$\mathbf{O} = \mathbf{X} \mathbf{W} + \mathbf{b}$$

where $\mathbf{W}$ is the weight matrix, $\mathbf{X}$ is the input, and $\mathbf{b}$ is the bias term. 

RNNs build upon this idea by introducing a hidden state, $\mathbf{H_t}$​, which acts as the model's memory. Instead of making a prediction solely from the current input $\mathbf{X}$, the model also incorporates information accumulated from previous timesteps. At timestep $t$, the hidden state $\mathbf{H}_t$ depends on both the current input $\mathbf{X_t}$​ and the previous hidden state $\mathbf{H_{t-1}}$:

$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_\textrm{xh} + \mathbf{H}_{t-1} \mathbf{W}_\textrm{hh} + \mathbf{b}_\textrm{h})$$

This variable $\mathbf{H_t}$ is what is capturing and retaining the sequence's historical information up to its current time step. Since this computation repeats itself at each time-step, it is deemed *recurrent*. 

For time $t$, the output of the output layer can be accomplished similarly to the computation in the MLP:

$$ \mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{\textrm{hq}} + \mathbf{b}_\textrm{q} $$

There are now a number of new model paramaters to keep track of:
- $\phi$ represents the nonlinear activation function being applied. In a standard RNN, this is typically `tanh()`, though other functions such as ReLU may also be used.
- $\mathbf{W_{\textrm{xh}}}$ is the weight matrix that maps the current input vector $\mathbf{X_t}$ to the hidden state.
- $\mathbf{W_{\textrm{hh}}}$ is the recurrent weight matrix that maps the previous hidden state $\mathbf{H_{t-1}}$ to the current hidden state.
- $\mathbf{W_{\textrm{hq}}}$ is the output weight matrix that maps the hidden state to the output vector.
- $\mathbf{b_{\textrm{h}}}$ and $\mathbf{b_{\textrm{q}}}$ are the bias vectors associated with the hidden-state computation and output computation, respectively.

You have seen `ReLU` and `sigmoid` before, so below we visualize the function graph of `tanh` compared to the other functions(left) and a transformation of data when `tanh` is applied to a sample of data (right). 

In [ ]:
# 1. Define the continuous range for plotting the curves
x = np.linspace(-4, 4, 200)
y_tanh = np.tanh(x)
y_sigmoid = 1 / (1 + np.exp(-x))
y_relu = np.maximum(0, x)

# Plot 1: Comparing Activation Functions
plt.subplot(1, 2, 1)
plt.plot(x, y_tanh, label=r'$\tanh(x)$', color='blue', linewidth=2)
plt.plot(x, y_sigmoid, label=r'$\sigma(x)$ (Sigmoid)', color='orange', linestyle='--', linewidth=1.5)
plt.plot(x, y_relu, label='ReLU(x)', color='green', linestyle=':', linewidth=1.5)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.axhline(y=0, color='k', linewidth=0.5)
plt.axvline(x=0, color='k', linewidth=0.5)
plt.title('Activation Functions Comparison')
plt.xlabel('Input ($x$)')
plt.ylabel('Output')
plt.ylim(-1.2, 2.0)
plt.legend()

# Plot 2: Visualizing Sample Data Transformation
plt.subplot(1, 2, 2)
plt.plot(x, y_tanh, color='blue', alpha=0.3, label=r'$\tanh(x)$ curve')

# Define discrete sample data points (e.g., hidden states or input features)
sample_inputs = np.array([-3.0, -1.5, -0.5, 0.0, 0.5, 1.5, 3.0])
sample_outputs = np.tanh(sample_inputs)

# Plot sample points along the X-axis (before transformation) and Y-axis (after transformation)
plt.scatter(sample_inputs, np.zeros_like(sample_inputs), color='red', zorder=5, label='Inputs ($x$)')
plt.scatter(np.zeros_like(sample_outputs), sample_outputs, color='purple', zorder=5, label='Outputs (tanh($x$))')

# Draw tracking lines mapping input -> curve -> output
for si, so in zip(sample_inputs, sample_outputs):
    plt.plot([si, si, 0], [0, so, so], color='gray', linestyle=':', alpha=0.6)

plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.axhline(y=0, color='k', linewidth=0.5)
plt.axvline(x=0, color='k', linewidth=0.5)
plt.title('Sample Data Transformation')
plt.xlabel('Input Value')
plt.ylabel('Output Value')
plt.ylim(-1.2, 1.2)
plt.legend(loc='upper left')

plt.tight_layout()

To implement our RNN, we begin with the `__init__` method. As before, this is where we define the learnable parameters of the model. Referring to the equations above, we need parameters that transform the current input, carry information through the hidden state, and produce the final output prediction.

Based on the RNN equations, you will need parameters corresponding to the input-to-hidden transformation ($\mathbf{W_{\textrm{xh}}}$​), the hidden-to-hidden recurrent transformation ($\mathbf{W_{\textrm{hh}}}$​), and the hidden-state bias ($\mathbf{b_{\textrm{h}}}$​). Once the hidden state has been computed, it must be mapped to the model output. To accomplish this, create a linear output layer, which introduces an additional output weight matrix ($\mathbf{W_{\textrm{hq}}}$​) and bias ($\mathbf{b_{\textrm{q}}}$​).

- **Q2.1:** Complete the `__init__` implementation in the code block below. Explain how you made use of parameter `num_hiddens`. 

- **A2.1:** 

In [ ]:
class RNNClassifier_v1(d2l.Classifier):
    def __init__(self, input_size=5, num_hiddens=64, num_outputs=5, lr=0.01, pos_weight=None):
        super().__init__()
        self.save_hyperparameters()

        # Manually initialize parameters


# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

        # Store a parameter that represents an increase in the loss associated with false negatives
        self.pos_weight = pos_weight

Also note the addition of a new parameter, `pos_weight`. Recall that the predictive maintenance dataset is highly imbalanced: most examples correspond to healthy machine operation, while failures are relatively rare. As a result, a model may achieve high accuracy by simply predicting the majority class (no failure) for every example.

To address this issue, we can assign a larger penalty to mistakes involving the positive class. In binary classification, the `pos_weight` parameter increases the loss associated with false negatives, encouraging the model to pay greater attention to failure examples during training. This helps prevent the model from converging to a trivial solution that always predicts the majority class and instead encourages it to learn meaningful patterns from the data.

The `forward()` method will require a bit more work than the models we have built previously. Begin by initializing the hidden state, `H`, as a tensor of zeros. Then, process the input sequence one timestep at a time. At each timestep, extract the current input vector and update the hidden state using the recurrent update equation discussed above.

Use the hidden state from the previous timestep together with the current input to compute the new hidden state, applying `torch.tanh()` as the activation function. By repeatedly updating `H`, the model can accumulate information from earlier observations and use it to inform later predictions. Once the entire sequence has been processed, the final hidden state can be passed to the output layer to generate a prediction.

- **Q2.2:** Complete the `forward` implementation in the code block below. You will need to initialize the hidden state `H`, process the input sequence one time-step `t` at a time by looping through `seq_len`, and at each time-step, update the hidden state using the recurrent update equation provided above. 

    Explain how in your code your hidden state `H` is initialized, and where it is updated for the time steps in your sequence. 

- **A2.2:** 

In [ ]:
@d2l.add_to_class(RNNClassifier_v1) 
def forward(self, X, H=None):
        batch_size, seq_len, _ = X.shape

        # Initialize the hidden state to zeros if not provided

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

        # Use the updated hidden state to compute the output logits for multi-label classification
        return torch.matmul(H, self.W_hy) + self.b_y

When we first encountered binary classification in Lab-02, we introduced Binary Cross Entropy (BCE) loss, a specialization of cross-entropy loss for problems with two possible outcomes. Although the predictive maintenance dataset contains multiple failure labels, each label represents an independent binary classification task, as either the corresponding failure occurred or it did not.

When inheriting from `d2l.Classifier`, the default loss function is standard multiclass cross entropy. Since our task consists of independent binary predictions, we will override this behavior and instead use PyTorch's `nn.BCEWithLogitsLoss`. This loss function combines a sigmoid activation with binary cross entropy in a numerically stable implementation, making it well suited for multi-label binary classification problems such as this one.


This is also where we need to address the issue of class imbalance. Earlier, we introduced the parameter `pos_weight`, which is used specifically with binary classification losses. To understand its effect, let's revisit the mathematical definition of BCE loss:

$$\text{Loss} = - [ y \cdot \log \hat{y}_i + (1 - y) \cdot \log (1 - \hat{y}_i) ]$$

When a positive class weight, $\mathbf{p}$, is introduced, the loss becomes:

$$\text{Loss} = - [ \mathbf{p} \cdot y \cdot \log \hat{y}_i + (1 - y) \cdot \log (1 - \hat{y}_i) ]$$

Notice that the weight $\mathbf{p}$ is applied only to the term associated with positive examples ($y=1$). As a result, mistakes involving the positive class contribute more heavily to the overall loss, encouraging the model to pay greater attention to rare failure events.


- **Q2.3:** To understand the effect of `pos_weight`, simplify the loss function for the two possible label values: $y=0$ and $y=1$. How does $\mathbf{p}$, corresponding to the `pos_weight` parameter, affect the loss in both cases? Why is this useful for highly imbalanced datasets such as the predictive maintenance dataset?

- **A2.3:** 

The `loss()` implementation is provided below. Notice that we pass the tensor `self.pos_weight` to the `pos_weight` argument of `BCEWithLogitsLoss`, allowing PyTorch to automatically apply the weighting during training.

In [ ]:
@d2l.add_to_class(RNNClassifier_v1)
def loss(self, y_hat, y):
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
    return loss_fn(y_hat, y)

One final step before training: when using the `Classifier` class from the `d2l` framework, the default implementation assumes a multiclass classification problem. For the same reason that we overrode the `loss()` function to use binary cross entropy, we also need to override the `accuracy()` function.

The default implementation expects a matrix of logits, with one output for each class. In our case, however, our model produces a single logit per example, whereas the default `d2l` accuracy function expects a vector of logits corresponding to multiple classes. To avoid this issue, we will compute the predictions manually by thresholding the output logits and then comparing the resulting labels against the ground-truth targets element by element.

Below, we implement this custom accuracy calculation using the `preds` and `correct` tensors.

In [ ]:
@d2l.add_to_class(RNNClassifier_v1)
def accuracy(self, Y_hat, Y, averaged=True):
        preds = (Y_hat > 0.0).float()
        correct = (preds == Y).float()
        return correct.mean() if averaged else correct.sum()

We copy the helper method we have been using to print out our accuracy and loss values:

In [ ]:
def print_model_results(board):
    # Print out final accuracies and loss values:
    final_tr_loss  = float(board.data['train_loss'][-1][1])
    final_val_loss = float(board.data['val_loss'][-1][1])
    final_val_acc  = float(board.data['val_acc'][-1][1])

    print("\n" + "="*40)
    print(f"Final Training Loss:     {final_tr_loss:.4f}")
    print(f"Final Validation Loss:   {final_val_loss:.4f}")
    print("-"*40)
    print(f"Final Validation Accuracy: {final_val_acc * 100:.2f}%")
    print("="*40)

Now, we can use the default `d2l.Trainer` to fit our model and data. We can calculate the penalty weights `pos_weight` for our labels by counting the negatives and positives across each class and dividing them. We add 1 to the `positives` in the denominator to avoid any issues of potentially dividing by 0, resulting in the equation `calculated_weights = negatives / (positives + 1)`.

In [ ]:
maintenance_df = pd.read_csv('data/ai4i2020.csv')
MaintenanceData = MaintenanceSequenceDataset(df=maintenance_df, seq_len=10, batch_size=64)

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE


model_1 = RNNClassifier_v1(input_size=5, num_hiddens=64, num_outputs=5, lr=0.05, pos_weight=calculated_weights)

# Fit pipeline
trainer_1 = d2l.Trainer(max_epochs=50, gradient_clip_val=1.0)
trainer_1.fit(model_1, MaintenanceData)

#Print results
print_model_results(model_1.board)

To determine whether the model is actually learning to identify failures, we need to examine its predictions in more detail. One of the most useful tools for this is the confusion matrix, which compares the model's predictions against the true labels. By visualizing the confusion matrix as a heatmap, we can quickly identify where the model performs well and where it struggles.

The helper functions below define a general-purpose `plot_confusion_matrix()` utility that can be used throughout this lab. Confusion matrices provide a much more complete picture of classification performance than accuracy alone, allowing us to evaluate how well the model detects each failure type, identify common misclassifications, and assess whether our techniques for handling class imbalance were successful.

In [ ]:
def _plot_multiclass(all_targets, all_preds, class_names, title):
    num_classes = len(class_names)
    cm = np.zeros((num_classes, num_classes), dtype=int)

    for true_label, pred_label in zip(all_targets, all_preds):
        true_idx = int(true_label)
        pred_idx = int(pred_label)

        if 0 <= true_idx < num_classes and 0 <= pred_idx < num_classes:
            cm[true_idx, pred_idx] += 1


    if class_names is None:
        class_names = [f"Class {i}" for i in range(cm.shape[0])]
        

    fig, ax = plt.subplots(figsize=(7, 5.5))

    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)

    ax.set_xticks(np.arange(num_classes))
    ax.set_yticks(np.arange(num_classes))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)

    # Annotate counts
    threshold = cm.max() / 2 if cm.max() > 0 else 0

    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > threshold else "black",
            )

    plt.tight_layout()
    plt.show()


def _plot_multilabel_grid(all_targets, all_preds, class_names, title):

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(title, fontsize=15, fontweight='bold')

    axes = axes.flatten()

    xlabels = ['Pred 0\n(Normal)', 'Pred 1\n(Fail)']
    ylabels = ['True 0\n(Normal)', 'True 1\n(Fail)']

    for i in range(min(len(class_names), all_targets.shape[1])):

        # Build 2x2 confusion matrix manually
        y_true = all_targets[:, i]
        y_pred = all_preds[:, i]

        tn = np.sum((y_true == 0) & (y_pred == 0))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        tp = np.sum((y_true == 1) & (y_pred == 1))

        cm = np.array([[tn, fp], [fn, tp]])

        ax = axes[i]
        im = ax.imshow(cm, cmap='Reds')
        ax.set_title(f'{class_names[i]}', fontsize=11, fontweight='bold'
        )

        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])

        ax.set_xticklabels(xlabels)
        ax.set_yticklabels(ylabels)

        ax.set_xlabel('Predicted Label', fontsize=9)
        ax.set_ylabel('True Label', fontsize=9)

        # Annotate cells
        threshold = cm.max() / 2 if cm.max() > 0 else 0

        for r in range(2):
            for c in range(2):
                ax.text(c, r, str(cm[r, c]), ha='center', va='center',
                    color='white' if cm[r, c] > threshold else 'black', fontsize=11)

    # Hide the unused sixth subplot
    axes[5].axis('off')

    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(model, data_module, class_names, mode='multiclass', title=None):
    all_preds = []
    all_targets = []
    
    val_dataloader = data_module.get_dataloader(train=False)
    
    # Universal extraction loop
    with torch.no_grad():
        for batch in val_dataloader:
            X, Y = batch[0], batch[1]
            logits = model(X)
            
            if mode == 'multiclass':
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.numpy())
                all_targets.extend(Y.numpy())
            elif mode == 'multilabel':
                preds = (torch.sigmoid(logits) > 0.5).int()
                all_preds.append(preds.numpy())
                all_targets.append(Y.numpy())
                
    # Delegate out to the proper plotting matrix engine
    if mode == 'multiclass':
        _plot_multiclass(all_targets, all_preds, class_names, title)
    elif mode == 'multilabel':
        _plot_multilabel_grid(np.vstack(all_targets), np.vstack(all_preds), class_names, title)

Now, we call this on our from-scratch RNN and predictive maintenance data to better visualize how it actually performed:

In [ ]:
failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
title = 'AI4I Multi-Label Predictive Maintenance Grid Analysis'

plot_confusion_matrix(model_1, MaintenanceData, class_names=failure_modes, mode='multilabel', title=title)

- **Q2.4:** Recall that `pos_weight` was introduced to help counter class imbalance by increasing the penalty for missed failures. Based on the confusion matrices, was this strategy successful? Which failure modes exhibited the highest false negative rates, and what might explain their difficulty?

- **A2.4:**

At this point, we have implemented a fully functional RNN from scratch. Compared to CNNs, the implementation is relatively straightforward, making it easier to understand the core mechanics of sequence modeling. However, as with previous labs, it is often more practical to rely on PyTorch's optimized implementations once the underlying concepts are understood.

PyTorch provides a built-in recurrent layer through `nn.RNN`, eliminating the need to manually manage hidden-state updates. We will pair this with an `nn.Linear` layer to map the final hidden state to our five output classes. As before, we will also incorporate class weights through `pos_weight` to help address the severe class imbalance present in the dataset.

In [ ]:
class RNNClassifier_v2(d2l.Classifier):
    def __init__(self, input_size=1, num_hiddens=64, num_outputs=5, lr=0.001, pos_weight=None):
        super().__init__()
        self.save_hyperparameters()
        
        # Concise Built-in RNN Layer
        # batch_first=True expects inputs with shape: (batch_size, seq_len, input_size)
        self.rnn = torch.nn.RNN(input_size, num_hiddens, num_layers=1, batch_first=True)
        
        # Fully connected layer to map hidden states to the 5 output classes
        self.linear = torch.nn.Linear(num_hiddens, num_outputs)


        if pos_weight is not None:
            # Setting it as an attribute allows PyTorch to track it smoothly
            self.pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
            self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        else:
            self.loss_fn = torch.nn.BCEWithLogitsLoss()

@d2l.add_to_class(RNNClassifier_v2)
def loss(self, Y_hat, Y):
    return self.loss_fn(Y_hat, Y)

@d2l.add_to_class(RNNClassifier_v2)
def accuracy(self, Y_hat, Y, averaged=True):
        preds = (Y_hat > 0.0).float()
        correct = (preds == Y).float()
        return correct.mean() if averaged else correct.sum()

The `forward()` function is considerably more concise when using `nn.RNN`. Rather than explicitly looping through the sequence and updating the hidden state at each timestep, we can pass the entire sequence directly to the recurrent layer. PyTorch performs these operations internally using optimized implementations, reducing both code complexity and execution time.

In [ ]:
@d2l.add_to_class(RNNClassifier_v2)
def forward(self, X):
        # out shape: (batch_size, seq_len, num_hiddens)
        # h_n shape: (num_layers, batch_size, num_hiddens)
        out, h_n = self.rnn(X)
        
        # Grab the hidden state from the very last timestep of the sequence
        final_time_step = out[:, -1, :]
        
        # Project to output scores
        return self.linear(final_time_step)

- **Q2.5:** Complete the code block below to train our concise implementation (`RNNClassifier_v2`) using the same hyperparameters as before. How does the accuracy and computational performance compare to our from-scratch version?

- **A2.5:**

In [ ]:
# Instantiate the model with the same hyperparameters as before, but now using the concise RNN implementation

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE


### 3. Activity Recognition from Motion

So far, we have transformed tabular data into sequences. We now move to datasets where the temporal structure is intrinsic to the measurements themselves.

The Heterogeneity Human Activity Recognition ([HHAR](https://archive.ics.uci.edu/dataset/344/heterogeneity+activity+recognition)) dataset, collected by the UCI Machine Learning repository, contains sensor data collected from smartphones and smartwatches carried by 30 participants as they performed a variety of everyday activities. By recording measurements over time while subjects walked, sat, stood, climbed stairs, and performed other common actions, the dataset provides a rich example of real-world sequential data.

Although the full HAR dataset includes measurements derived from multiple device types and sensors (e.g. gyroscope, magnetometer, GPS), we will focus exclusively on the accelerometer data in this lab. An [accelerometer](https://en.wikipedia.org/wiki/Accelerometer) measures acceleration along three spatial axes: $x$, $y$, and $z$. Each data sample therefore consists of a sequence of three-dimensional acceleration measurements collected over time, along with a label indicating the activity being performed during that interval. Our goal will be to use these sensor readings to correctly classify the subject's activity.

In [ ]:
har_df = pd.read_csv('data/HAR_accelerometer.csv')
har_df.head()

The three primary features in this dataset are accelerometer measurements along the `x`, `y`, and `z` axes. These readings, measured in $m/s^2$, capture how the smartphone's motion changes over time in three-dimensional space. As different activities produce distinct movement patterns, the resulting acceleration signals vary considerably between actions such as walking, sitting, or climbing stairs.

The label column, `gt`, identifies the activity being performed by the individual wearing the device at each point in time. This gives us another classification problem: using a sequence of acceleration measurements, determine which activity generated the observed motion pattern. Unlike the tabular datasets we have explored previously, the temporal evolution of the signal is critical to making an accurate prediction.

Below, a histogram of the class distribution is provided: 

In [ ]:
print("Activity Class Distribution:")
print(har_df['gt'].value_counts())

# Sum the occurrences of each activity type
activity_counts = har_df['gt'].value_counts()

# Plot the distribution
plt.figure(figsize=(10, 5))
plt.bar(activity_counts.index, activity_counts.values, color=plt.cm.viridis(np.linspace(0, 1, len(activity_counts.values))), alpha=0.7)

plt.title('HAR Activity Label Distribution')
plt.xlabel('Activity Class')
plt.ylabel('Total Sensor Rows')
plt.xticks(rotation=45)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Unlike the predictive maintenance dataset, the HAR dataset is relatively well balanced, with examples distributed across several activity classes. In addition, every sample belongs to exactly one activity, making this a standard multiclass classification problem.

Another important difference is that the HAR dataset includes measurements collected over time, giving it a natural sequential structure. Rather than treating each observation independently, we can analyze how the accelerometer readings evolve from one timestep to the next. To build intuition for this temporal behavior, let's visualize several segments of the raw accelerometer signals.

Complete the implementation of the function `plot_activity()` below. The `x`, `y`, and `z` data are all on the same scale and can be plotted together in simple line plots using `plt.plot()`. We then call this function on each class, giving us insight into the trends our model may look for across these value readings. 

In [ ]:
def plot_activity(df, activity_name, num_samples=500):
    # Filter for the target activity
    activity_df = df[df['gt'] == activity_name].iloc[:num_samples]

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

# Run the visualizer for comparison
plot_activity(har_df, 'walk')
plot_activity(har_df, 'sit')
plot_activity(har_df, 'stand')
plot_activity(har_df, 'bike')
plot_activity(har_df, 'stairsup')
plot_activity(har_df, 'stairsdown')

- **Q3.1:** Accelerometer readings provide linear acceleration  ($a(t) = \frac{d^2x}{dt^2}$, or the second derivative of position with respect to time).  At first glance, it may seem that recovering a person's path would be straightforward: if acceleration is the second derivative of position, then position can be obtained by integrating twice ($x(t) = \iint a(t) \,dt$). Why would reconstructing a user's path with pure integration be difficult here? How does an RNN solve this problem?

- **A3.1:** 

- **Q3.2:** Examine the accelerometer signals for the sitting and standing activities. Although the individual is largely stationary, the sensor readings are not all zero. What physical phenomenon produces this persistent baseline acceleration? Why does this acceleration appear on one or more of the sensor axes even when the subject is not moving?

- **Q3.2:** 

Now that we have a better understanding of the time-series data we are working with, we are ready to construct our dataset class. The first step is deciding how to divide the continuous sensor stream into sequences that can be processed by the model. Rather than making a prediction from a single accelerometer reading, we will provide the model with a window of recent measurements and allow it to learn patterns that develop over time.

To accomplish this, we must implement a sliding-window transformation that converts the original 2nd-order array of samples and features into a 3rd-order tensor of sequences, timesteps, and features. Because the dataset consists of a continuous stream of sensor readings, we will partition it into shorter fixed-length segments, each of which serves as a single training example for our RNN.

Complete the helper method `create_windows()` to generate fixed-length sequences from the continuous sensor stream. Your function should iterate through the dataset using a sliding-window approach. The loop should stop once there are not enough remaining samples to form a complete sequence, and the stride parameter should determine how far the window advances after each iteration.

For each position of the window, extract:
- `window`: the sequence of sensor measurements corresponding to the current time interval.
- `target`: the activity label associated with that sequence.

Append each `window` and target to the lists `X_windows` and `y_windows`, respectively. Once all sequences have been generated, convert these lists into tensors and return them so they can be used within the dataset's `__init__` method. Note that this implementation will be quite similar to how you built the sequences for the Predictive Maintenance datset earlier, partitioned into a helper method here to keep our `__init__` block more concise. The key difference here, and the biggest thing to watch out for, is that we are introducing `stride` as a parameter, which we saw in Lab-04. Recall the general form of a python `for` loop is `(start, end, step)`. Often in these labs, we start from zero and can just define the end point, looping over its range. However, we now want to define the third position, `step`, as our `stride` value. Therefore, it is important that you define all three positional arguments of the `for` condition, such that python interprets them properly. 

In [ ]:
def create_windows(features, labels, seq_len=50, stride=25):
    # Segment streams into temporal sequence windows
    X_windows = []
    y_windows = []
        
# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE
    return X_tensor, y_tensor
        

With that helper method defined, the rest of our `__init__` method can be completed in a similar fashion to what we did previously.

- **Q3.3:** Finish the `HARDataset` class in the code block below. You will need to extract and standardize the features, and map the labels to integer values before calling `create_windows()`. Recall from previous labs that `pd.factorize()` can be used to return the integer-coded labels and classes in one tuple. Explain / defend the steps in your implementation. 

- **A3.3:** 

In [ ]:
class HARDataset(d2l.DataModule):
    def __init__(self, df, seq_len=200, stride=20, batch_size=64, max_rows=500000):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Filter out unlabeled transitional periods ("null" class)
        df = df[df['gt'] != 'null'].dropna()

        # 2. Extract input dimensions
        # 3. Standardize features
        # 4. Map labels to integer codes

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

        self.num_classes = len(self.classes_)

        # Call our helper method
        self.X, self.y = create_windows(features, labels, seq_len, stride)
        
        # 5. Apply an 80/20 train/validation index separation boundary
        total_samples = self.X.shape[0]
        self.num_train = int(total_samples * 0.8)
        self.num_val = total_samples - self.num_train
        
        print(f"Total processed sequence samples: {total_samples}")
        print("X Input Feature Block Shape:", self.X.shape)
        print("y Target Matrix Label Shape:", self.y.shape)

    def get_dataloader(self, train):
        i = slice(0, self.num_train) if train else slice(self.num_train, None)
        return self.get_tensorloader((self.X, self.y), train, i)

Let's confirm that our dataloader is working properly:

In [ ]:
har_df = pd.read_csv('data/HAR_accelerometer.csv')
HARData= HARDataset(df=har_df, seq_len=200, batch_size=64)
X, y = next(iter(HARData.train_dataloader()))
print(X.shape, y.shape)

### 4. Long-Short Term Memory (LSTM) Models

We now turn to a more sophisticated recurrent architecture: the *Long Short-Term Memory (LSTM)* network. In the vanilla RNNs we have studied so far, the hidden state $h_t$​ serves as both the model's current time-step representation and its memory of past observations. While this allows information to persist across timesteps, standard RNNs often struggle to capture long-range dependencies because gradients can gradually vanish as they are propagated backward through many recurrent updates.

LSTMs address this limitation by introducing a second state, the cell state $C_t$​. While the hidden state continues to represent the model's current time-step, the cell state acts as a dedicated memory pathway that can carry information across many timesteps. Through a series of carefully designed gating operations, the network learns what information should be remembered, updated, and exposed to the next layer. This structure helps preserve useful information over longer time horizons and makes training substantially more stable than in a vanilla RNN.

The cell state is controlled by three gates: the *forget* gate, the *input* gate, and the *output* gate. The forget gate determines how much information from the previous cell state should be retained. The input gate controls how much new information from the current timestep should be written into memory. Finally, the output gate determines how much of the internal memory should be exposed through the hidden state $h_t$​.

To understand how an LSTM operates, we will examine each of these gates in turn. Each gate produces values between 0 and 1 using a sigmoid activation function. Values near 0 suppress the flow of information, while values near 1 allow information to pass largely unchanged. Although the cell state is often described as a "conveyor belt" carrying information through time, the gates themselves are not applied sequentially. Instead, they are typically computed in parallel from the current input $x_t$​ and previous hidden state $h_{t-1}$​, and their outputs are then combined to update the cell state and hidden state.


The input gate, $I_t$, determines how much new information from the current timestep should be written into memory:

$$I_t = \sigma(X_t W_\textrm{xi} + H_{t-1} W_\textrm{hi} + b_\textrm{i})$$

The forget gate, $F_t$, controls how much of the previous memory should be retained / erased.

$$F_t = \sigma(X_t W_\textrm{xf} + H_{t-1} W_\textrm{hf} + b_\textrm{f})$$

Finally, the output gate, $O_t$, controls how much of the internal memory should be exposed through the hidden state:

$$O_t = \sigma(X_t W_\textrm{xo} + H_{t-1} W_\textrm{ho} + b_\textrm{o})$$

Before the cell state can be updated, the model generates a candidate memory ($\tilde{C}_t$)​, representing new information that could potentially be stored:

$$\tilde{\mathbf{C}}_t = \tanh(\mathbf{X}_t \mathbf{W}_\textrm{xc} + \mathbf{H}_{t-1} \mathbf{W}_\textrm{hc} + \mathbf{b}_\textrm{c})$$

Here, the tanh activation produces values on the interval $(−1,1)$, allowing the candidate memory to add or subtract information from the existing cell state.

The cell state is then updated by combining retained information from the previous timestep with newly generated candidate information:


$$C_t = (F_t \odot C_{t-1}) + (I_t \odot \tilde{C}_t)$$

The operator $\odot$ denotes the *Hadamard product*, or element-wise multiplication. This allows the forget and input gates to independently control the contribution of each element of the cell state.

Finally, the hidden state is computed from the updated cell state. The cell state is first passed through a tanh activation and then filtered by the output gate:

$$H_t = O_t \odot \tanh(C_t)$$

While these equations may initially appear intimidating, they are largely extensions of the recurrent network we implemented earlier. The key difference is that instead of maintaining a single hidden state, the LSTM uses gates to regulate the flow of information into, out of, and through a dedicated memory cell. In practice, these gates are often implemented efficiently using a small number of large matrix operations. With the mathematical foundations in place, we can now begin defining the `__init__` function and constructing our LSTM layer.

Finish the `__init__` implementation below; the parameters for the input gate are provided for you as a reference, and you should define the forget gate, output gate, and candidate memory parameters in a similar fashion. While tedious, this can help you to understand the place of each piece within our architecture, of which will need to be properly kept track of throughout the network. 

In [ ]:
class LSTM_layer(d2l.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()

        # INPUT GATE PARAMETERS 
        self.W_xi = torch.nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.W_hi = torch.nn.Parameter(torch.randn(num_hiddens, num_hiddens) * sigma)
        self.b_i  = torch.nn.Parameter(torch.zeros(num_hiddens))

        # FORGET GATE PARAMETERS
        # OUTPUT GATE PARAMETERS
        # CANDIDATE MEMORY PARAMETERS

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE


The `forward()` method is where the core LSTM logic is implemented. Here, we compute the input, forget, and output gates, generate the candidate memory, and update both the cell state and hidden state. Unlike the vanilla RNN, which only maintains a hidden state, the LSTM tracks two pieces of information throughout the sequence: the hidden state `H` and the cell state `C`. As a result, both states must be updated at every timestep.

The initialization of the hidden state `H` and cell state `C` has been provided for you. To complete the `forward()` method, iterate through the input sequence one timestep at a time. Within the loop, compute the input gate (`I`), forget gate (`F`), output gate (`O`), and candidate memory (`C_tilde`). Each of these quantities is computed from the current input and previous hidden state using an affine transformation followed by a nonlinear activation function. The gates use a `sigmoid` activation, while the candidate memory uses `tanh`.

Use `torch.matmul()` to perform the necessary matrix multiplications, add the appropriate bias terms, and apply the corresponding activation functions. Once the gates have been computed, update the cell state and hidden state using the LSTM equations presented above. After processing the entire sequence, return the final hidden state and cell state.

- **Q4.1:** Why do the input, forget, and output gates use a sigmoid activation, while the candidate memory uses a tanh activation?

- **A4.1:** 

In [ ]:
@d2l.add_to_class(LSTM_layer)
def forward(self, inputs, H_C=None):
    # inputs shape: (batch_size, seq_len, num_inputs) -> e.g., [64, 200, 3]
    batch_size, seq_len, _ = inputs.shape

    # Initialize hidden state (H) and memory cell state (C) if not provided
    if H_C is None:
        H = torch.zeros((batch_size, self.num_hiddens))
        C = torch.zeros((batch_size, self.num_hiddens))
    else:
        H, C = H_C

    outputs = []

    # Loop across sequence
    for t in range(seq_len):

        # Extract the 3D feature slice for time-step t: shape (batch_size, num_inputs)
        X = inputs[:, t, :]

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

        outputs.append(H)
        
    # Stack the sequence outputs along dim 1: (batch_size, seq_len, num_hiddens)
    outputs_tensor = torch.stack(outputs, dim=1)
    
    # Only the hidden state is passed to the next time-step, but we return both H and C for potential use in the next sequence
    return outputs_tensor, (H, C)

- **Q4.2:** Suppose the forget gate always outputs values very close to 0. What would you expect to happen to the cell state and the model's ability to learn long-term dependencies?

- **A4.2:** 

Implementing the `forward()` method is largely an exercise in carefully translating the LSTM equations into code. The main challenge is keeping track of the many weights, biases, gates, and state updates involved. This complexity is one of the primary drawbacks of implementing recurrent architectures from scratch.

To verify that your implementation is correct, run the unit test below before proceeding. A passing test will help confirm that the gate computations and state updates have been implemented properly.

In [ ]:
# Testing the forward pass of the LSTM_layer implementation with dummy data
batch_size = 64
seq_len = 200
num_inputs = 3      # x, y, z accelerometer axes
num_hiddens = 64    # Your internal hidden layer dimension

# Create dummy input data matching our DataLoader's exact shape
mock_batch = torch.randn(batch_size, seq_len, num_inputs)
print(f"Testing input shape: {mock_batch.shape}")

model_step = LSTM_layer(num_inputs=num_inputs, num_hiddens=num_hiddens)

# Run a forward pass
outputs_tensor, (H, C) = model_step(mock_batch)

# Verify the output shapes
print("\n--- Verification ---")
print(f"Outputs Tensor Shape:  {outputs_tensor.shape}   <- Expected: [64, 200, 64]")
print(f"Final Hidden State (H): {H.shape}            <- Expected: [64, 64]")
print(f"Final Cell State (C):   {C.shape}            <- Expected: [64, 64]")

assert outputs_tensor.shape == (batch_size, seq_len, num_hiddens), "Outputs tensor shape mismatch!"
assert H.shape == (batch_size, num_hiddens), "Final hidden state shape mismatch!"
assert C.shape == (batch_size, num_hiddens), "Final cell state shape mismatch!"

print("\nSuccess! All tensor dimensions align perfectly across all 200 time-steps.")

We have now successfully implemented an LSTM layer, a significant improvement over the vanilla recurrent layer. However, the LSTM alone is not a complete classification model. While it can process a sequence and produce hidden representations, it still needs a final output layer to map those representations to class predictions.

Fortunately, the difficult part is already behind us. We now simply need to wrap the LSTM layer inside a classifier architecture by adding a linear output head. This classifier will inherit from `d2l.Classifier`, giving us access to the training framework, loss function, optimizer, and evaluation utilities we have used throughout the course. The result will be a complete end-to-end model capable of learning from sequential data and producing activity classifications.

In [ ]:
class LSTMClassifier_v1(d2l.Classifier):
    def __init__(self, num_inputs, num_hiddens, num_classes, lr=0.001):
        super().__init__()
        self.save_hyperparameters()
        
        # Instantiate your custom elementary scratch LSTM
        self.lstm = LSTM_layer(num_inputs, num_hiddens)
        # Linear output layer mapping hidden state to class logits
        self.fc = torch.nn.Linear(num_hiddens, num_classes)
        
    def forward(self, X):
        # 1. Forward through the custom LSTM loop
        # out shape: [batch_size, seq_len, num_hiddens]
        out, (H, C) = self.lstm(X)
        
        # 2. Extract the hidden state at the final time-step
        final_timestep_hidden = out[:, -1, :] # Shape: [batch_size, num_hiddens]
        
        # 3. Project to class logits for CrossEntropyLoss
        return self.fc(final_timestep_hidden) # Shape: [batch_size, num_classes]
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

We are now ready to train our from-scratch LSTM. 

In [ ]:
har_df = pd.read_csv('data/HAR_accelerometer.csv')
HARData = HARDataset(df=har_df, seq_len=200, batch_size=64)

model_3 = LSTMClassifier_v1(num_inputs=3, num_hiddens=64, num_classes=HARData.num_classes, lr=0.001)

trainer_3 = d2l.Trainer(max_epochs=10)
trainer_3.fit(model_3, HARData)

print_model_results(model_3.board)

Lets take a look at a confusion matrix of the true and predicted labels. We can call the universal `plot_confusion_matrix` function already defined above to do so; it just needs the HAR data's labels passed in as a list.

In [ ]:
plot_confusion_matrix(model_3, HARData, class_names=HARData.classes_, mode='multiclass', title='HAR Activity Recognition Confusion Matrix')

- **Q4.3:** Which activities are classified most accurately, and which are the most challenging for the model? What characteristics of these activities might explain the difference?

- **A4.3:** 

Our concise implementation can be accomplished in a single step, as we can use `torch.nn.LSTM` to define the LSTM layer. 

In [ ]:
class LSTMClassifier_v2(d2l.Classifier):
    def __init__(self, num_inputs, num_hiddens, num_classes, lr=0.001):
        super().__init__()
        self.save_hyperparameters()
        
        # Concise LSTM layer
        self.lstm = torch.nn.LSTM(input_size=num_inputs, hidden_size=num_hiddens, batch_first=True)
        
        # Fully connected classification head
        self.fc = torch.nn.Linear(num_hiddens, num_classes)
        
    def forward(self, X):
        # nn.LSTM returns two things:
        # 1. All hidden states across the sequence: [batch_size, seq_len, num_hiddens]
        # 2. A tuple of the final states: (last_hidden_H, last_cell_C)
        output, (H, C) = self.lstm(X)
        
        # Pull the hidden state from the very last time-step of the sequence
        final_timestep_hidden = output[:, -1, :] # Shape: [batch_size, num_hiddens]
        
        # Project it to class logits
        return self.fc(final_timestep_hidden) # Shape: [batch_size, num_classes]

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

- **Q4.4:** Explain the line of code, `final_timestep_hidden = output[:, -1, :]`. How does passing this directly into our final linear classifier head successfully produce the class prediction logits?

- **A4.4:** 

In [ ]:
model_4 = LSTMClassifier_v2(num_inputs=3, num_hiddens=64, num_classes=HARData.num_classes, lr=0.001)

trainer_4 = d2l.Trainer(max_epochs=10)
trainer_4.fit(model_4, HARData)

print_model_results(model_4.board)

In [ ]:
plot_confusion_matrix(model_4, HARData, class_names=har_labels, mode='multiclass', title='HAR Activity Recognition Confusion Matrix')

- **Q4.5:** In our concise implementation above, we make sure to set about `batch_first=true` as a parameter for our LSTM layer. Briefly review the [torch.nn.LSTM](https://docs.pytorch.org/docs/2.12/generated/torch.nn.LSTM.html) documentation. Why was this step necessary?

- **A4.4:** 

In the cell below, investigate how network *depth* affects performance on the HAR classification task. The `num_layers` parameter of [torch.nn.LSTM](https://docs.pytorch.org/docs/2.12/generated/torch.nn.LSTM.html) controls the number of stacked LSTM layers, while `dropout` specifies the fraction of activations that are randomly zeroed during training to help reduce overfitting.

Experiment with both parameters by either extending `LSTMClassifier_v2` to support configurable depth and dropout, or by creating a new `LSTMClassifier_v3` class. 

In [ ]:
# Implement / explore a deeper LSTM architecture with multiple stacked layers and dropout regularization to improve generalization and performance on the HAR dataset.

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE


- **Q4.6:** As you varied these settings, how did training time, validation accuracy, and generalization performance change? Does increasing the number of layers consistently improve performance, and does dropout help prevent overfitting?

- **A4.5:** 

### 5. Physiological Signal Analysis

Thus far, we have constructed sequences from tabular measurements and analyzed continuous sensor streams. We now examine physiological signals, where the waveform itself contains diagnostic information.

Electrocardiogram ([ECG](https://en.wikipedia.org/wiki/Electrocardiography)) recordings measure the electrical activity of the heart and contain rich temporal structure that reflects the state of a biological system. Rather than identifying a human activity, our goal is to determine whether the observed waveform corresponds to normal or abnormal cardiac behavior. This introduces a new challenge for recurrent models: learning to recognize subtle waveform patterns and dependencies across multiple heartbeats.

We have downloaded and pre-processed the [MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0/). This dataset contains 8 half-hour excerpts of two-channel ambulatory ECG recordings collected from 47 subjects studied by a Boston-area hospital between 1975 and 1979. 

In [ ]:
ecg_df = pd.read_csv("data/mitbih_train.csv")
ecg_df.head()

The goal of this dataset is to classify different types of cardiac arrhythmias from ECG recordings. If you inspect the dataset, you will notice that the feature columns have generic numeric names and the class labels are encoded as integer values rather than descriptive categories. The final column contains the target label, with classes indexed from 0 to 4:
- `0` - **N** (**Normal**): A normal heartbeat.
- `1` - **S** (**Supraventricular**): A premature heartbeat originating in the upper chambers of the heart.
- `2` - **V** (**Ventricular**): A premature heartbeat originating in the ventricles. These beats can be associated with more - serious cardiac conditions.
- `3` - **F** (**Fusion**): A hybrid beat formed when a normal beat and a ventricular beat occur simultaneously.
- `4` - **Q** (**Unknown**): A beat that does not clearly fit into the other categories.

To make the dataset easier to work with, replace the scientific-notation feature column names with simple numerical indices and rename the final column to `label`.

In [ ]:
# Convert the columns to a range of integers for easier handling, and rename the last column to "label" for clarity

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

print("Last 5 columns after the cleanup:")
print(ecg_df.columns[-5:])
ecg_df.head()

As we always do when working with multi-class data, we visualize the class distribution to check the label balance:

In [ ]:
# Define the 5 specific heartbeat classes for the MIT-BIH dataset
ecg_labels = ['N (Normal)', 'S (Supraventricular)', 'V (Ventricular)', 'F (Fusion)', 'Q (Unknown)']
label_counts = ecg_df['label'].value_counts().sort_index()
print(f"Active heartbeat cases per class:\n{label_counts.to_string()}\n")

# Create the bar chart
plt.figure(figsize=(10, 6))
plt.bar(label_counts.index, label_counts.values, color=plt.cm.viridis(np.linspace(0, 1, len(label_counts.values))), alpha=0.7)

# Add values on top of each bar for explicit clarity
for i, val in enumerate(label_counts.values):
    plt.text(i, val + 2, str(val), ha='center', va='bottom', fontweight='bold')

plt.title('Distribution of Heartbeat Classes', fontsize=14)
plt.xticks(label_counts.index, ecg_labels, rotation=0, fontsize=10)
plt.xlabel('Heartbeat Type', fontsize=12)
plt.ylabel('Total Number', fontsize=12)
plt.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


Once again, we are dealing with a heavily imbalanced dataset. Most ECG recordings correspond to normal heartbeats, while several arrhythmia classes occur much less frequently. Techniques such as class weighting and confusion-matrix analysis will, once again, be essential for properly evaluating our models.

Before returning to model development, it is worth taking a closer look at the problem domain itself. One particularly important class is label `2` ([Ventricular arrhythmia](https://en.wikipedia.org/wiki/Arrhythmia#Ventricular_arrhythmia)). These beats originate in the ventricles, the lower chambers of the heart, and occur prematurely relative to the normal cardiac rhythm. Because ventricular arrhythmias can be associated with serious cardiac conditions, accurately detecting them is far more important than simply maximizing overall accuracy. This highlights an important ethical consideration in machine learning: when models are deployed in domains such as healthcare, mistakes can have real consequences. A well-designed model may help clinicians identify patients who need further attention, while an unreliable model could lead to missed diagnoses or unnecessary interventions.

To better understand the characteristics of ventricular beats, we can visualize the ECG recordings directly. In the heatmap below, each row corresponds to a different heartbeat, while color indicates signal amplitude. By stacking many examples together, we can begin to identify recurring waveform patterns and observe how ventricular beats differ from normal cardiac activity.

In [ ]:
# Split into classes
normal_beats = ecg_df[ecg_df['label'] == 0].iloc[:500, :-1].values
ventricular_beats = ecg_df[ecg_df['label'] == 2].iloc[:500, :-1].values

fig, axes = plt.subplots(2, 2, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})

vmin = min(normal_beats.min(), ventricular_beats.min())
vmax = max(normal_beats.max(), ventricular_beats.max())

im1 = axes[0,0].imshow(normal_beats, aspect='auto', cmap='hot', vmin=vmin, vmax=vmax
)

axes[0,0].set_title("Normal Beats (Class N)")
axes[0,0].set_xlabel("Time Sample")
axes[0,0].set_ylabel("Heartbeat Example")

im2 = axes[0,1].imshow(ventricular_beats, aspect='auto', cmap='hot', vmin=vmin, vmax=vmax)

axes[0,1].set_title("Ventricular Beats (Class V)")
axes[0,1].set_xlabel("Time Sample")
axes[0,1].set_ylabel("Heartbeat Example")

# Shared colorbar
fig.colorbar(im2, ax=axes[0,:], label="Signal Amplitude")

normal_mean = normal_beats.mean(axis=0)
ventricular_mean = ventricular_beats.mean(axis=0)

axes[1,0].plot(normal_mean, color='tab:blue', linewidth=2)
axes[1,0].set_title("Average Normal Beat")
axes[1,0].set_xlabel("Time Sample")
axes[1,0].set_ylabel("Amplitude")

axes[1,1].plot(ventricular_mean, color='tab:red', linewidth=2)
axes[1,1].set_title("Average Ventricular Beat")
axes[1,1].set_xlabel("Time Sample")
axes[1,1].set_ylabel("Amplitude")

plt.show()


- **A5.1:** Compare the normal (N) and ventricular (V) heartbeat visualizations. What differences do you observe in the shape, timing, or amplitude of the dominant peaks? Are these differences consistent across examples? Explain how a sequence model such as an LSTM might use these patterns to distinguish between the two heartbeat classes.

- **A5.1:**

Lets now compare what the average heartbeat patterns look like across all label classes.

Finish the code below to plot the waveform data for each label's average heartbeat patterns. Calculate the mean (like before) as well as the standard deviation, and then use the combination of `plt.plot()` and [fill_between()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.fill_between.html) to visualize the data representing 1 standard deviation around the mean line. 

In [ ]:
plt.figure(figsize=(14, 8))
time_steps = range(187)

for i, label in enumerate(ecg_labels):

    # Filter for a specific heartbeat type
    # Calculate mean and standard deviation across time steps
    # Plot the mean waveform with a shaded area representing one standard deviation above 
    # and below the mean (set alpha=0.1 for transparency)

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

plt.title("Average ECG Waveforms by Heartbeat Type")
plt.xlabel("Time-step (1/125th of a second)")
plt.ylabel("Normalized Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

- **Q5.2:** The "Unknown" (Q) class exhibits some of the largest and most dramatic waveform peaks in the dataset. Intuitively, it may seem that the most extreme heart activity should be the easiest to diagnose. Why, then, do you think these heartbeats were classified as Unknown rather than assigned to a specific arrhythmia category?

- **A5.2:**

We now return to building our machine learning pipeline. As always, the first step is constructing our dataset class and implementing its `__init__` method. 

Unlike the HAR dataset, where we created sequences from a continuous stream of sensor readings, the ECG dataset is already organized into fixed-length sequences. Each row corresponds to a complete heartbeat waveform, so no additional windowing is required. Our main task is simply to separate the waveform samples into `X` and the heartbeat class labels into `y`, and store both in a format suitable for model training.

In [ ]:
class ECGDataset(d2l.DataModule):
    def __init__(self, df, batch_size=64):
        super().__init__()
        self.save_hyperparameters()

        df.columns = list(range(len(df.columns)))
        df = df.rename(columns={187: "label"})

        # Here we can randomize the order of the dataset to ensure that training and validation splits 
        # are not biased by any inherent ordering in the original CSV file. 
        # This is particularly important for time-series data where consecutive rows may be correlated.
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        
        X_raw = df.drop(columns=["label"]).values  
        y_raw = df["label"].values                 

        self.X = torch.tensor(X_raw, dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(y_raw, dtype=torch.long)

        total_samples = self.X.shape[0]
        self.num_train = int(total_samples * 0.8)
        self.num_val = total_samples - self.num_train

    def get_dataloader(self, train):
        i = slice(0, self.num_train) if train else slice(self.num_train, None)
        return self.get_tensorloader((self.X, self.y), train, i)

We test our dataloaders before continuing:

In [ ]:
ecg_df = pd.read_csv("data/mitbih_train.csv")
ECGdata = ECGDataset(df=ecg_df, batch_size=64)

X, y = next(iter(ECGdata.train_dataloader()))
print(X.shape, y.shape)

### 6. Gated Recurrent Units (GRUs)

Our next architecture, the *Gated Recurrent Unit (GRU)*, is a streamlined alternative to the LSTM. The primary goal of the GRU is to achieve performance comparable to an LSTM while using fewer parameters and requiring less computation. As the name suggests, GRUs still rely on gates to control the flow of information through the network. However, whereas an LSTM uses three gates (input, forget, and output) along with a separate cell state, a GRU uses only two gates: a reset gate and an update gate, and maintains only a single hidden state.

Like the gates in an LSTM, both GRU gates use a sigmoid activation function, producing values on the interval $(0,1)$. These values determine how much information should be retained, discarded, or incorporated into the hidden state at each timestep.

The *reset gate*, $\mathbf{R}_t$, controls how strongly the previous hidden state influences the candidate hidden state. It is defined as:

$$\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_\textrm{xr} + \mathbf{H}_{t-1} \mathbf{W}_\textrm{hr} + \mathbf{b}_\textrm{r})$$

When elements of the reset gate are close to 0, the corresponding portions of the previous hidden state are suppressed. When they are close to 1, the model is free to leverage more of its past memory. In this sense, the reset gate determines how much historical information should contribute to the next state computation.

The *update gate*, $\mathbf{Z}_t$, combines many of the responsibilities of the input and forget gates found in an LSTM. Its role is to determine how much of the previous hidden state should be preserved versus how much new information should be incorporated. The update gate is computed as:

$$\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_\textrm{xz} + \mathbf{H}_{t-1} \mathbf{W}_\textrm{hz} + \mathbf{b}_\textrm{z})$$

Notice that both gates depend on the current input $\mathbf{X}_t$ and the previous hidden state $\mathbf{H}_{t-1}$. Since neither gate depends on the output of the other, they can be computed independently and are typically evaluated in parallel by modern deep learning frameworks.

Once these gates have been calculated, the model computes a *candidate hidden state*:

$$\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_\textrm{xh} + (\mathbf{R}_t \odot \mathbf{H}_{t-1}) \mathbf{W}_\textrm{hh} + \mathbf{b}_\textrm{h})$$

The key term is $(\mathbf{R}_t \odot \mathbf{H}_{t-1})$, where the reset gate is applied element-wise to the previous hidden state. If a component of the reset gate is close to 0, the corresponding historical information is largely ignored when constructing the candidate state. If it is close to 1, that information is carried forward. This gives the GRU the ability to decide when past information remains useful and when it should rely more heavily on the current input.

Finally, the hidden state is updated according to:

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1} + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t$$

Here, the update gate acts as a weighted blend between the previous hidden state and the newly computed candidate state. If $\mathbf{Z}_t$​ is close to 1, the model primarily retains information from the past. If it is close to 0, the model favors the newly computed candidate state. In this way, the update gate gives the GRU direct control over how quickly its internal memory evolves over time.


- **Q6.1:** Suppose our ECG sequence contains a long period of stable behavior followed by a sudden change. Based on the hidden-state update equation, how might the update gate respond during these two phases?

- **A6.1:** 

We can implement this from scratch similar to how we first implemented the LSTM layer from scratch, by initializing all of our parameters, weights and biases in our `__init__` block and taking care of the gate, candidate hidden state and final hidden state calculations in our `forward` block.

Finish the `__init__` implementation below.

In [ ]:
class GRU_layer(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.num_hiddens = num_hiddens
                
# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE


For the `forward()` function, you can follow a very similar approach to the LSTM implementation. The overall structure remains the same: initialize the hidden state, iterate through the sequence one timestep at a time, compute the gate values, and update the hidden state accordingly. The primary challenge is carefully translating the GRU equations into code and ensuring that each weight matrix and bias term is applied in the correct location.


- **Q6.2:** Finish the `forward()` implementation in the following code block. Below, briefly explain how you handle iterating over a sequence, defining the reset and update gate, and calculate the candidate and final hidden state.

- **A6.2:** 

In [ ]:
@d2l.add_to_class(GRU_layer)
def forward(self, X):
        batch_size, seq_len, _ = X.shape
        
        # Initialize the single tracking hidden state (H) to zeros
        H = torch.zeros((batch_size, self.num_hiddens))
        outputs = []
        
        # The Temporal Core Loop
        for t in range(seq_len):

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE
        outputs_tensor = torch.cat(outputs, dim=1)
        
        # Return both the full sequence outputs and the final state tuple 
        return outputs_tensor, H

We can test the implementation of our GRU layer using a similar set of output shape test cases as we did with the LSTM. 

In [ ]:
# Define dimensions matching our standard sequence setup
batch_size = 64
seq_len = 187
num_inputs = 1
num_hiddens = 64

# Instantiate the isolated from-scratch GRU layer
scratch_layer = GRU_layer(num_inputs, num_hiddens)

# Force all parameters to be entirely deterministic 
# (1.0 for weights, 0.0 for biases to strip out all random variance)
for name, param in scratch_layer.named_parameters():
    if 'W' in name:
        torch.nn.init.constant_(param, 1.0)
    elif 'b' in name:
        torch.nn.init.constant_(param, 0.0)

# Create a uniform mock input sequence of all 0.5s
mock_sequence_batch = torch.full((64, 187, 1), 0.5)

# Run the isolated forward pass
with torch.no_grad():
    outputs_tensor, final_H = scratch_layer(mock_sequence_batch)

# Tensor Shape Verification
print(f"Input Batch Shape:     {list(mock_sequence_batch.shape)} (Expected: [64, 187, 1])")
print(f"Outputs Tensor Shape:  {list(outputs_tensor.shape)} (Expected: [64, 187, 64])")
print(f"Final Hidden (H) Shape:{list(final_H.shape)} (Expected: [64, 64])")


We have now built a fully functional GRU layer from scratch, giving us a detailed understanding of how gated recurrent units operate internally. Since this lab is already fairly extensive, we will forgo training the custom implementation. As we saw with LSTM, because our from-scratch implementation relies on explicit Python `for` loops, training would be considerably slower than practical implementations. 

Since we will be training this model on our ECG heartbeat dataset, we need to address the class imbalance in the training data. In our predictive maintenance example, we handled imbalance using the `pos_weight` parameter of `BCEWithLogitsLoss`. That solution works well for binary classification tasks but cannot be directly applied to a multi-class problem.

For multi-class classification, `CrossEntropyLoss` provides a weight parameter that allows us to assign a weight to each class. We will therefore add a `class_weights` parameter to our model's `__init__()` method and pass these weights to the loss function. This causes errors on underrepresented classes to contribute more heavily to the overall loss, helping the model avoid becoming biased toward the majority classes.

In [ ]:
class GRUClassifier_v1(d2l.Classifier):
    def __init__(self, num_inputs, num_hiddens, num_classes, class_weights=None, lr=0.001):
        super().__init__()
        self.save_hyperparameters()
        
        # batch_first=True expects incoming tensor shape: [batch_size, seq_len, num_inputs]
        self.gru = torch.nn.GRU(input_size=num_inputs, hidden_size=num_hiddens, batch_first=True)        
        self.fc = torch.nn.Linear(num_hiddens, num_classes)
        
        # Handle imbalance with Weighted Cross Entropy
        if class_weights is not None:
            self.loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float))
        else:
            self.loss_fn = torch.nn.CrossEntropyLoss()
        
    def forward(self, X):
        # Pass the  sequence batch straight into  GRU kernel
        outputs_tensor, _ = self.gru(X)
        
        # Slice out the hidden state from the very last time-step along the sequence axis
        # final_timestep_hidden shape: [batch_size, num_hiddens]
        final_timestep_hidden = outputs_tensor[:, -1, :]
        
        # Project directly to class logits
        return self.fc(final_timestep_hidden)

    def loss(self, Y_hat, Y):
        return self.loss_fn(Y_hat, Y)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

A common choice for our `CrossEntropyLoss()` weights is to set the `weight` for each class inversely proportional to the number of training examples in that class. This causes mistakes on rare classes to incur a larger penalty than mistakes on common classes. In practice, however, extremely imbalanced datasets may benefit from less aggressive weighting schemes, such as using the inverse square root of the class counts, to prevent the rarest classes from dominating the training process.

- **Q6.3:** For our ECG dataset, should class weights be determined solely by how often a class appears in the dataset, or should they also reflect the consequences of making mistakes on that class?

- **A6.3:** 

- **Q6.4:** In the following code block, compute an appropriate set of `class_weights` for use with `CrossEntropyLoss`. You may use inverse frequency weighting, inverse square-root weighting, or a custom scheme tailored to this ECG classification problem. Also, you will want to normalize your class weights such that the average class weight is $1.0$. Remember that class weights can reflect not only how rare a heartbeat class is, but also how important it is to correctly identify that class. Defend your choice below and discuss the tradeoffs involved in your weighting strategy.

- **A6.4:** 

In [ ]:
# Class labels from before
ecg_labels = ['Normal (N)', 'Supraventricular (S)', 'Ventricular (V)', 'Fusion (F)', 'Unknown (Q)']

# We may or may not have labeled the dataframe, so just grab the 187th column
class_counts = ecg_df.iloc[:, 187].value_counts().sort_index()

# Convert to tensor for PyTorch compatibility
class_counts_tensor = torch.tensor(class_counts.values, dtype=torch.float32)

# YOUR CODE STARTS HERE

# YOUR CODE ENDS HERE

print("Class weights: ")
print(class_weights)


Now we train our model, making use of our `class_weights`:

In [ ]:
ECGdata = ECGDataset(df=ecg_df, batch_size=64)
model_6 = GRUClassifier_v1(num_inputs=1, num_hiddens=64, class_weights=class_weights, num_classes=5, lr=0.001)
trainer_6 = d2l.Trainer(max_epochs=10)
trainer_6.fit(model_6, ECGdata)

print_model_results(model_6.board)

Now, let's apply our confusion matrix function to the ECG dataset and the concise GRU model to gain a deeper understanding of the model's performance beyond overall accuracy.

In [ ]:
plot_confusion_matrix(model_6, ECGdata, class_names=ecg_labels, mode='multiclass')

- **Q6.5:** Not all classification errors have the same consequences. Using the confusion matrix, identify the most concerning mistakes made by the model. Which heartbeat classes would you most want the model to distinguish correctly in a real clinical setting, and why? Discuss whether the model's performance is acceptable given the types of errors it makes.

- **A6.5:** 

### Lab Submission
When you are ready to submit this assignment on Canvas, be sure to save this file first, then run the following command to generate the two files you'll need to upload for Canvas submission:

In [ ]:
!bash ../3250_create_submission.sh

### Prelab Questions

- **Q0.1)\:** Both MLPs and RNNs use the term *hidden*, but in different ways. What is the difference between a hidden layer and a hidden state, and what role does each play in its respective model?

- **A0.1)\:** A hidden layer in an MLP is *hidden* because the weights for the forward pass are learned and "hidden" to us as trainers. Hidden states in RNNs are similar as its hidden as the recurrent state (context and sequence history) is learned and not defined. Generally similar for the hidden part but encode different data.

- **Q0.2)\:** Suppose for our RNN model we set `seq_len = 1`. In what sense does an RNN become similar to a standard feed-forward neural network?


- **A0.2)\:** It would become similar as it wouldn't actually be recurrent and would effectively just do the standard forward pass without refeeding. (is refeeding a proper term?)

- **Q0.3)\:** Both RNNs and LSTMs are designed for sequential data. What additional component does an LSTM include that is not present in a vanilla RNN, and what purpose does it serve?

- **A0.3)\:** The LSTM has a "Cell State" which allows it to remember or forget information selectively via gating which allows the LSTM to remember or forget over longer sequences, compared to RNNs which can struggle to learn longer sequences.